# MotorCam – fáze 6: experimenty do práce
1. Veřejná data vs. doladění na českých záběrech (povrch i detektor)
2. Analýza chyb podle podmínek (stín, noc, mokro, rozmazání) + ukázkové snímky
3. Rychlost, doporučená rychlost a varování v čase

Vše se ukládá do `results/figures` (PNG + PDF) a `results/tables` (CSV).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/motorcam'
REPO_URL = 'https://github.com/MiniGue1/motorcam.git'   # u soukromého repa: https://<token>@github.com/...

import os
%cd /content
if not os.path.exists('motorcam'):
    !git clone -q $REPO_URL motorcam
%cd /content/motorcam
!pip install -q -r requirements.txt
# archivy, vlastní data, modely a výsledky trvale na Google Drive
for d in ['data/raw/rdd2022/archives', 'data/raw/rscd/archives', 'data/custom', 'models', 'results']:
    os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True); os.makedirs(os.path.dirname(d) or '.', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf $d
        os.symlink(f'{DRIVE_DIR}/{d}', d)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENÍ – Runtime → Change runtime type → T4 GPU')

In [ ]:
# RSCD z fáze 1 (archiv na Drive), vlastní anotace z Label Studia
RSCD_ARCHIVE = f'{DRIVE_DIR}/data/raw/rscd/archives/rscd.zip'   # uprav podle názvu
!python -m src.dataset.download rscd --archive "$RSCD_ARCHIVE"
!python -m src.dataset.rscd_to_classes
import os
if os.path.exists('data/custom/labelstudio_export.json'):
    !python -m src.dataset.import_annotations labelstudio data/custom/labelstudio_export.json

In [ ]:
# data z fáze 1 (archivy už jsou na Drive – stáhne jen chybějící, pak rozbalí a převede)
!python -m src.dataset.download rdd2022
!python -m src.dataset.rdd_to_yolo

## 1a) Povrch: veřejná vs. doladěný model

In [ ]:
!python -m src.experiments.compare surface \
    --model verejna=models/surface_public/best.pt --model doladeny=models/surface_finetuned/best.pt \
    --test RSCD=data/processed/rscd_cls --test ceske=data/processed/custom_cls --device cuda
from IPython.display import Image, display
display(Image('results/figures/porovnani_povrch.png')); display(Image('results/figures/porovnani_povrch_tridy.png'))

## 1b) Detektor: veřejná vs. doladěný model

In [ ]:
!python -m src.experiments.compare detector \
    --model verejna=models/det_public/weights/best.pt --model doladeny=models/det_finetuned/weights/best.pt \
    --test RDD2022=models/det_public/dataset/data.yaml:val --test ceske=data/processed/custom_yolo/data.yaml:test
display(Image('results/figures/porovnani_detektor.png'))

## 2) Analýza chyb podle podmínek

In [ ]:
!python -m src.experiments.errors --pred results/tables/povrch_doladeny_ceske_test_predikce.csv \
    --meta data/processed/custom_meta.csv --det models/det_finetuned/weights/best.pt --det-data data/processed/custom_yolo
display(Image('results/figures/chyby_podminky.png'))
import os
if os.path.exists('results/figures/chyby_ukazky.png'): display(Image('results/figures/chyby_ukazky.png'))

## 3) Rychlost a varování před zatáčkami (z dema, fáze 5)

In [ ]:
!python -m src.experiments.speed_plot results/demo/demo_timeline.csv --name jizda1
display(Image('results/figures/rychlost_jizda1.png'))

## Složení dat (fáze 1)

In [ ]:
!python -m src.dataset.dataset_stats
import glob
for f in sorted(glob.glob('results/figures/data_*.png')): display(Image(f))